# Phase 1 - Preprocess BCI Competition IV-2a

Convert raw `BCI_Competition_Data/A0xT.mat` files into per-subject `Processed_BCI_Competition_Data/<subject>/{trials.npz, metadata.csv}` plus a `manifest.json` and `loso_splits.json` consumed by every downstream method notebook.

Window: `[tmin=2.0s, tmax=6.0s]` relative to trial onset, artifact-marked trials dropped, first 22 EEG channels kept (last 3 EOG channels discarded). Self-contained - no project imports.

## 1. Imports and configuration

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.io import loadmat
from tqdm.auto import tqdm

REPO_ROOT = Path.cwd().resolve()
while REPO_ROOT.name != 'new_repository' and REPO_ROOT.parent != REPO_ROOT:
    REPO_ROOT = REPO_ROOT.parent
if REPO_ROOT.name != 'new_repository':
    raise RuntimeError('Run this notebook from inside the new_repository tree.')

RAW_DIR = REPO_ROOT / 'BCI_Competition_Data'
PROCESSED_DIR = REPO_ROOT / 'Processed_BCI_Competition_Data'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

TMIN_S = 2.0
TMAX_S = 6.0
DROP_ARTIFACTS = True
MAT_GLOB = 'A*T.mat'

BCI_IV_2A_EEG_CHANNELS = [
    'Fz', 'FC3', 'FC1', 'FCz', 'FC2', 'FC4', 'C5', 'C3', 'C1', 'Cz', 'C2', 'C4',
    'C6', 'CP3', 'CP1', 'CPz', 'CP2', 'CP4', 'P1', 'Pz', 'P2', 'POz',
]
CLASS_ID_TO_NAME = {1: 'left_hand', 2: 'right_hand', 3: 'feet', 4: 'tongue'}

print('REPO_ROOT      =', REPO_ROOT)
print('RAW_DIR        =', RAW_DIR)
print('PROCESSED_DIR  =', PROCESSED_DIR)

REPO_ROOT      = C:\Users\unnat\Documents\GitHub\EEGFeatureExtraction\new_repository
RAW_DIR        = C:\Users\unnat\Documents\GitHub\EEGFeatureExtraction\new_repository\BCI_Competition_Data
PROCESSED_DIR  = C:\Users\unnat\Documents\GitHub\EEGFeatureExtraction\new_repository\Processed_BCI_Competition_Data


c:\Users\unnat\Desktop\EEGFeatureExtraction\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Run extraction helpers

In [2]:
def iter_runs(mat_path: Path):
    data = loadmat(mat_path, squeeze_me=True, struct_as_record=False)['data']
    if not isinstance(data, np.ndarray):
        data = np.array([data], dtype=object)
    for run_idx, run in enumerate(data):
        yield run_idx, run


def extract_trials_from_run(run, subject_id, run_idx, tmin_s, tmax_s, drop_artifacts):
    x = np.asarray(run.X, dtype=np.float32)
    y = np.asarray(run.y).reshape(-1).astype(np.int32)
    onsets = np.asarray(run.trial).reshape(-1).astype(np.int64)
    artifacts = np.asarray(run.artifacts).reshape(-1).astype(np.int32)
    fs = int(run.fs)
    if y.size == 0 or onsets.size == 0:
        return None, None

    start_off = int(round(tmin_s * fs))
    end_off = int(round(tmax_s * fs))
    windows, records = [], []
    for i, onset in enumerate(onsets):
        label_id = int(y[i])
        artifact = int(artifacts[i]) if i < artifacts.size else 0
        if drop_artifacts and artifact == 1:
            continue
        s, e = int(onset + start_off), int(onset + end_off)
        if s < 0 or e > x.shape[0]:
            continue
        windows.append(x[s:e, :22].T)
        records.append({
            'subject': subject_id,
            'run_index': run_idx,
            'trial_index_in_run': i,
            'label_id': label_id,
            'label_name': CLASS_ID_TO_NAME.get(label_id, f'class_{label_id}'),
            'artifact': artifact,
            'start_sample': s,
            'end_sample': e,
            'sampling_rate': fs,
        })
    if not windows:
        return None, None
    return np.stack(windows, axis=0), pd.DataFrame.from_records(records)


def preprocess_subject(mat_path: Path, output_dir: Path):
    subject_id = mat_path.stem
    all_x, all_meta = [], []
    for run_idx, run in iter_runs(mat_path):
        x_run, meta_run = extract_trials_from_run(
            run, subject_id, run_idx, TMIN_S, TMAX_S, DROP_ARTIFACTS
        )
        if x_run is None:
            continue
        all_x.append(x_run)
        all_meta.append(meta_run)
    if not all_x:
        return None
    x_all = np.concatenate(all_x, axis=0).astype(np.float32)
    meta_df = pd.concat(all_meta, ignore_index=True)
    y_all = meta_df['label_id'].to_numpy(dtype=np.int32)
    subj_out = output_dir / subject_id
    subj_out.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(
        subj_out / 'trials.npz',
        X=x_all,
        y=y_all,
        channel_names=np.array(BCI_IV_2A_EEG_CHANNELS, dtype=object),
        class_names=np.array([CLASS_ID_TO_NAME[i] for i in sorted(CLASS_ID_TO_NAME)], dtype=object),
    )
    meta_df.to_csv(subj_out / 'metadata.csv', index=False)
    return {
        'subject': subject_id,
        'n_trials': int(x_all.shape[0]),
        'n_channels': int(x_all.shape[1]),
        'n_samples': int(x_all.shape[2]),
        'sampling_rate': int(meta_df['sampling_rate'].iloc[0]),
    }

## 3. Run preprocessing across all subjects (with tqdm progress)

In [4]:
if not RAW_DIR.is_dir():
    raise FileNotFoundError(
        f'Raw .mat directory not found: {RAW_DIR}. Place A0xT.mat files there before running.'
    )

mat_files = sorted(RAW_DIR.glob(MAT_GLOB))
if not mat_files:
    raise FileNotFoundError(
        f'No .mat files in {RAW_DIR} matching {MAT_GLOB!r}. Expecting BCI Competition IV-2a A0xT.mat files.'
    )

summaries = []
for mat_path in tqdm(mat_files, desc='Preprocessing subjects', unit='subj'):
    summary = preprocess_subject(mat_path, PROCESSED_DIR)
    if summary is not None:
        summaries.append(summary)

if not summaries:
    raise RuntimeError('No trials were extracted; check raw .mat contents.')

summary_df = pd.DataFrame(summaries).sort_values('subject').reset_index(drop=True)
summary_df.to_csv(PROCESSED_DIR / 'summary.csv', index=False)
summary_df

Preprocessing subjects: 100%|██████████| 9/9 [00:30<00:00,  3.37s/subj]


,subject,n_trials,n_channels,n_samples,sampling_rate
0,A01T,273,22,1000,250
1,A02T,270,22,1000,250
2,A03T,270,22,1000,250
3,A04T,262,22,1000,250
4,A05T,262,22,1000,250
5,A06T,219,22,1000,250
6,A07T,271,22,1000,250
7,A08T,264,22,1000,250
8,A09T,237,22,1000,250


## 4. Build manifest.json and loso_splits.json

In [5]:
subjects = summary_df['subject'].tolist()
loso_splits = [
    {'test_subject': s, 'train_subjects': [t for t in subjects if t != s]}
    for s in subjects
]

manifest = {
    'input_dir': str(RAW_DIR),
    'output_dir': str(PROCESSED_DIR),
    'glob': MAT_GLOB,
    'window_seconds': {'tmin': TMIN_S, 'tmax': TMAX_S},
    'drop_artifacts': DROP_ARTIFACTS,
    'eeg_channels': BCI_IV_2A_EEG_CHANNELS,
    'classes': CLASS_ID_TO_NAME,
    'subjects_processed': subjects,
    'total_trials': int(summary_df['n_trials'].sum()),
    'loso_split_count': len(loso_splits),
}

(PROCESSED_DIR / 'manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
(PROCESSED_DIR / 'loso_splits.json').write_text(json.dumps(loso_splits, indent=2), encoding='utf-8')

print(f'Subjects processed : {len(subjects)}')
print(f'Total trials       : {manifest["total_trials"]}')
print(f'LOSO splits        : {len(loso_splits)}')
print('Saved              :', PROCESSED_DIR / 'manifest.json')
print('Saved              :', PROCESSED_DIR / 'loso_splits.json')

Subjects processed : 9
Total trials       : 2328
LOSO splits        : 9
Saved              : C:\Users\unnat\Documents\GitHub\EEGFeatureExtraction\new_repository\Processed_BCI_Competition_Data\manifest.json
Saved              : C:\Users\unnat\Documents\GitHub\EEGFeatureExtraction\new_repository\Processed_BCI_Competition_Data\loso_splits.json
